#### 1. Column Names
#### Display all column names. Identify columns having leading/trailing spaces and
#### rename all columns into a consistent snake_case format.

In [2]:
import pandas as pd
import re

df = pd.read_csv("C:\\Users\\rohit\\Downloads\\matplotlib_messy_sales_2000.csv")

[c for c in df.columns if c != c.strip()]

df.columns = [
    re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]+', '_', c.strip())).strip('_').lower()
    for c in df.columns
]

print("Cleaned columns:", df.columns.tolist())

Cleaned columns: ['customer_id', 'order_date', 'city', 'product_category', 'customer_segment', 'gender', 'sales_channel', 'age', 'units_sold', 'unit_price', 'discount', 'customer_rating', 'is_active', 'returned', 'payment_method', 'gross_sales', 'discount_amount', 'revenue', 'cost', 'profit']


#### 2. Duplicate Records
#### Find the number of duplicate rows. Remove the duplicate records and verify that no  duplicates remain.

In [3]:
df.duplicated().sum()

np.int64(20)

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.duplicated().sum()

np.int64(0)

#### 3. Missing Values
#### Display the missing-value count and missing-value percentage for every column.

In [6]:
df.isnull().sum()

customer_id          0
order_date           4
city                25
product_category    24
customer_segment     0
gender               0
sales_channel        0
age                 27
units_sold           4
unit_price          24
discount            28
customer_rating     25
is_active            0
returned             0
payment_method      25
gross_sales          0
discount_amount      0
revenue             25
cost                 0
profit               0
dtype: int64

In [7]:
missing_value = df.isnull().sum() /len(df)*100

missing_value

customer_id         0.00
order_date          0.20
city                1.25
product_category    1.20
customer_segment    0.00
gender              0.00
sales_channel       0.00
age                 1.35
units_sold          0.20
unit_price          1.20
discount            1.40
customer_rating     1.25
is_active           0.00
returned            0.00
payment_method      1.25
gross_sales         0.00
discount_amount     0.00
revenue             1.25
cost                0.00
profit              0.00
dtype: float64

#### 4. Missing Categorical Values
#### Handle missing values in city, product_category, and payment_method using an
#### appropriate strategy.

In [8]:
df[['city', 'product_category', 'payment_method']].isnull().sum()

city                25
product_category    24
payment_method      25
dtype: int64

#### 5. Missing Numerical Values
#### Handle missing values in age, unit_price, discount_pct, customer_rating, and
#### revenue. Explain why you selected mean, median, or another method.

In [9]:
cols = ['age', 'unit_price', 'discount', 'customer_rating', 'revenue']

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

df[cols].isnull().sum()

age                0
unit_price         0
discount           0
customer_rating    0
revenue            0
dtype: int64

#### 6. Date Cleaning
#### Convert order_date into a proper datetime format. Identify invalid date values and handle them appropriately.

In [10]:
df = df.copy()

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

print("Invalid dates:", df['order_date'].isna().sum())

df = df.dropna(subset=['order_date']).copy()

print(df['order_date'].head())

Invalid dates: 25
0   2025-05-05
1   2025-10-22
2   2025-04-24
3   2025-05-17
4   2025-08-30
Name: order_date, dtype: datetime64[ns]


#### 7. Age Cleaning
#### The age column contains values such as 25 years, 30 yrs, 45Y, and N/A. Extract the
#### numerical age and convert the column into a numeric datatype.

In [11]:
df['age'] = pd.to_numeric(
    df['age'].astype(str).str.extract(r'(\d+)')[0],
    errors='coerce'
)

print(df['age'].head())
print(df['age'].dtype)

0    36
1    38
2    65
3    34
4    41
Name: age, dtype: int64
int64


#### 8. Age Outliers
#### Identify unrealistic ages such as 120, 150, and 200. Detect them using the IQR
#### method and handle them appropriately.

In [12]:
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[(df['age'] < lower) | (df['age'] > upper)]
print("Outliers:", outliers['age'].tolist())

# Replace outliers with median age
median_age = df['age'].median()
df.loc[(df['age'] < lower) | (df['age'] > upper), 'age'] = median_age

print("Outliers handled.")

Outliers: [150, 120, 120, 120, 200, 120, 150, 150, 150, 120, 150, 120, 120]
Outliers handled.


#### 9. Unit Price Cleaning
#### The unit_price column contains values such as ₹15000, $20000, 25000 INR, and 10k.
#### Convert all valid values into a single numeric format.

In [13]:
# Convert unit_price to numeric
def clean_price(x):
    x = str(x).strip().lower()
    
    if 'k' in x:
        return float(x.replace('k', '')) * 1000
    
    x = x.replace('₹', '').replace('$', '').replace('inr', '').replace(',', '').strip()
    return pd.to_numeric(x, errors='coerce')

df['unit_price'] = df['unit_price'].apply(clean_price)

print(df['unit_price'].head())
print(df['unit_price'].dtype)

0    24235.73
1    16789.55
2    24579.97
3     6240.13
4    23491.23
Name: unit_price, dtype: float64
float64


#### 10. Unit Price Outliers
#### Detect extreme unit_price values using the IQR method. Compare the number of
#### outliers before and after treatment.

In [14]:
Q1 = df['unit_price'].quantile(0.25)
Q3 = df['unit_price'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_before = ((df['unit_price'] < lower) | 
                   (df['unit_price'] > upper)).sum()

median_price = df['unit_price'].median()
df.loc[(df['unit_price'] < lower) | 
       (df['unit_price'] > upper), 'unit_price'] = median_price

outliers_after = ((df['unit_price'] < lower) | 
                  (df['unit_price'] > upper)).sum()

print("Outliers before:", outliers_before)
print("Outliers after:", outliers_after)

Outliers before: 14
Outliers after: 0


#### 11. Units Sold Cleaning
#### Clean values such as 5 units, 10 pcs, twenty, N/A, and 3 and convert the column into a numeric datatype.

In [15]:
words = {'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4,
         'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10}

def clean_units(x):
    x = str(x).strip().lower()
    
    if x in words:
        return words[x]
    
    return pd.to_numeric(
        x.replace('units', '').replace('pcs', '').strip(),
        errors='coerce'
    )

df['units_sold'] = df['units_sold'].apply(clean_units)

print(df['units_sold'].head())
print(df['units_sold'].dtype)

0    8.0
1    5.0
2    2.0
3    9.0
4    1.0
Name: units_sold, dtype: float64
float64


#### 12. Discount Cleaning
#### Clean values such as 10%, 20 percent, 5 %, none, and N/A. Convert the final
##### discount_pct column into numeric values between 0 and 100.

In [16]:
df['discount'] = (
    df['discount']
    .astype(str)
    .str.lower()
    .str.replace('%', '', regex=False)
    .str.replace('percent', '', regex=False)
    .str.strip()
    .replace(['none', 'n/a', 'nan'], pd.NA)
)

df['discount'] = pd.to_numeric(df['discount'], errors='coerce')

print(df['discount'].head())
print(df['discount'].dtype)

0    22.91
1    10.59
2    36.22
3    20.44
4    10.48
Name: discount, dtype: float64
float64


#### 13. Discount Outliers
#### Identify invalid discount values such as 75, 90, and 120. Determine which values are unrealistic and handle them.

In [17]:
invalid = df[df['discount'] > 100]

print("Invalid discounts:", invalid['discount'].tolist())

# Replace invalid values with median
median_discount = df.loc[df['discount'] <= 100, 'discount'].median()

df.loc[df['discount'] > 100, 'discount'] = median_discount

print("Invalid values after treatment:",
      (df['discount'] > 100).sum())

Invalid discounts: [120.0, 120.0, 120.0, 120.0, 120.0]
Invalid values after treatment: 0


#### 14. City Standardization
##### Standardize inconsistent city values such as:
##### o Mumbai
##### o mumbai
##### o MUMBAI
##### o Mumbai
##### o Bangalore
##### o bangalore
##### o Hyderbad

##### Create one consistent set of city names.

In [18]:
df['city'] = df['city'].astype(str).str.strip().str.title()

# Correct spelling
df['city'] = df['city'].replace({
    'Hyderbad': 'Hyderabad'
})

print(df['city'].unique())

['Mumbai' 'Delhi' 'Chennai' 'Hyderabad' 'Pune' 'Bengaluru' 'Nan'
 'Bangalore']


#### 15. Product Category Cleaning
#### Standardize values such as Electronics, electronics, ELECTRONICS, Home and Kitchen,
#### home & kitchen, and Beauty .

In [19]:
df['product_category'] = df['product_category'].astype(str).str.strip().str.title()

df['product_category'] = df['product_category'].replace({
    'Home And Kitchen': 'Home & Kitchen'
})

print(df['product_category'].unique())

['Clothing' 'Home & Kitchen' 'Electronics' 'Sports' 'Beauty' 'Nan']


#### 16. Gender Cleaning
#### Standardize values such as Male, male, Male, FEMALE, female , M, and F into consistent categories.

In [20]:
df['gender'] = df['gender'].astype(str).str.strip().str.lower()

df['gender'] = df['gender'].replace({
    'm': 'Male',
    'f': 'Female',
    'male': 'Male',
    'female': 'Female'
})

print(df['gender'].unique())

['Male' 'Female' 'other']


#### 17. Customer Segment Cleaning
#### Identify invalid values such as Prem, Reg, VIP, and Unknown. Map valid variations to the correct categories and decide how to handle invalid categories.

In [21]:
# Standardize customer segment
df['customer_segment'] = df['customer_segment'].astype(str).str.strip().str.lower()

df['customer_segment'] = df['customer_segment'].replace({
    'prem': 'Premium',
    'premium': 'Premium',
    'reg': 'Regular',
    'regular': 'Regular',
    'vip': 'VIP'
})

# Handle invalid values
valid = ['Premium', 'Regular', 'VIP']
df.loc[~df['customer_segment'].isin(valid), 'customer_segment'] = 'Unknown'

print(df['customer_segment'].value_counts())

customer_segment
Regular    1084
Premium     498
Unknown     393
Name: count, dtype: int64


#### 18. Binary Column Cleaning
#### Clean is_active and returned, where values contain combinations of:
#### o 0/1
#### o Yes/No
#### o Y/N
#### o Active/Inactive
#### o Returned/Not Returned

#### Convert both columns into a consistent 0/1 binary format.

In [23]:
# is_active
df['is_active'] = df['is_active'].astype(str).str.strip().str.lower().map({
    'yes': 1, 'y': 1, 'active': 1, '1': 1,
    'no': 0, 'n': 0, 'inactive': 0, '0': 0
})

# returned
df['returned'] = df['returned'].astype(str).str.strip().str.lower().map({
    'yes': 1, 'y': 1, 'returned': 1, '1': 1,
    'no': 0, 'n': 0, 'not returned': 0, '0': 0
})

print(df[['is_active', 'returned']].head())

   is_active  returned
0          1         0
1          1         0
2          1         0
3          1         0
4          1         0


#### 19. Data-Type Validation
#### After cleaning, display df.info() and verify that:
#### o Dates are datetime
#### o Numerical columns are numeric
#### o Binary columns contain only 0 and 1
#### o Categorical columns contain clean strings

In [25]:
# Check data types
df.info()

# Verify date
print("Date type:", df['order_date'].dtype)

# Verify numerical columns
num_cols = ['age', 'unit_price', 'discount',
            'customer_rating', 'revenue', 'units_sold']

print("\nNumerical columns:")
print(df[num_cols].dtypes)

# Verify binary columns
print("\nBinary values:")
print("is_active:", df['is_active'].unique())
print("returned:", df['returned'].unique())

# Verify categorical columns
cat_cols = ['city', 'product_category', 'gender', 'customer_segment',
            'payment_method']

print("\nCategorical columns:")
for col in cat_cols:
    print(col, ":", df[col].dropna().unique())

<class 'pandas.core.frame.DataFrame'>
Index: 1975 entries, 0 to 2019
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       1975 non-null   object        
 1   order_date        1975 non-null   datetime64[ns]
 2   city              1975 non-null   object        
 3   product_category  1975 non-null   object        
 4   customer_segment  1975 non-null   object        
 5   gender            1975 non-null   object        
 6   sales_channel     1975 non-null   object        
 7   age               1975 non-null   int64         
 8   units_sold        1967 non-null   float64       
 9   unit_price        1975 non-null   float64       
 10  discount          1975 non-null   float64       
 11  customer_rating   1975 non-null   float64       
 12  is_active         1975 non-null   int64         
 13  returned          1975 non-null   int64         
 14  payment_method    1950 non-nu

#### 20. Final Data Quality Check — Hardest
##### Perform a complete final audit of the cleaned dataset:
##### o Check duplicate rows
##### o Check missing values
##### o Check invalid dates
##### o Check numerical columns
##### o Check outliers
#####  o Check categorical inconsistencies
##### o Check binary values
##### o Check datatypes
##### o Verify that Revenue, Cost, and Profit are logically consistent

In [27]:
# 20. Final Data Quality Check

# 1. Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# 2. Missing values
print("\nMissing values:")
print(df.isnull().sum())

# 3. Invalid dates
print("\nInvalid dates:", df['order_date'].isna().sum())

# 4. Numerical columns
num_cols = ['age', 'unit_price', 'discount',
            'customer_rating', 'units_sold', 'revenue', 'cost', 'profit']

print("\nNumerical data types:")
print(df[num_cols].dtypes)

# 5. Outliers using IQR
print("\nOutliers:")
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) |
                (df[col] > Q3 + 1.5*IQR)).sum()
    print(col, ":", outliers)

# 6. Categorical inconsistencies
cat_cols = ['city', 'product_category', 'gender',
            'customer_segment', 'payment_method']

print("\nCategorical values:")
for col in cat_cols:
    print(col, ":", df[col].dropna().unique())

# 7. Binary values
print("\nBinary values:")
print("is_active:", df['is_active'].dropna().unique())
print("returned:", df['returned'].dropna().unique())

# 8. Datatypes
print("\nData types:")
print(df.dtypes)

# 9. Revenue, Cost and Profit consistency
print("\nRevenue/Cost/Profit check:")

# Expected: Profit = Revenue - Cost
df['profit_check'] = df['revenue'] - df['cost']

print("Incorrect profit rows:",
      (abs(df['profit'] - df['profit_check']) > 0.01).sum())

# Remove temporary column
df.drop(columns='profit_check', inplace=True)

Duplicate rows: 0

Missing values:
customer_id          0
order_date           0
city                 0
product_category     0
customer_segment     0
gender               0
sales_channel        0
age                  0
units_sold           8
unit_price           0
discount             0
customer_rating      0
is_active            0
returned             0
payment_method      25
gross_sales          0
discount_amount      0
revenue              0
cost                 0
profit               0
dtype: int64

Invalid dates: 0

Numerical data types:
age                  int64
unit_price         float64
discount           float64
customer_rating    float64
units_sold         float64
revenue            float64
cost               float64
profit             float64
dtype: object

Outliers:
age : 0
unit_price : 0
discount : 5
customer_rating : 0
units_sold : 15
revenue : 22
cost : 38
profit : 69

Categorical values:
city : ['Mumbai' 'Delhi' 'Chennai' 'Hyderabad' 'Pune' 'Bengaluru' 'Nan'
 'Bangalor